<a href="https://colab.research.google.com/github/alaaguedda/esg-sustainability-reports/blob/main/esg_report_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip uninstall -y bitsandbytes -q
!pip install -q "bitsandbytes==0.45.5" --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.9 MB/s eta 0:00:00


In [ ]:
# Stage 0.0: Environment bootstrap — installs pinned to versions known to work
# together on Colab Free T4 (CUDA 12.x) as of this project's start. Rationale
# for each package lives inline; nothing here is decorative.
#
# pdfplumber + pymupdf (fitz): redundant PDF backends. pymupdf is fast for
#   page-count checks (used during filtering, before we commit to full text
#   extraction); pdfplumber is used for the actual text extraction because it
#   preserves reading order / layout better for report-style PDFs.
# huggingface_hub: dataset file listing + streaming download without cloning
#   the whole repo (repo is large; we only want a filtered subset of PDFs).
# bitsandbytes + accelerate + transformers: 4-bit quantized local inference,
#   required to fit a 7B model in Colab Free's ~15GB T4 VRAM / 12GB RAM.
# sentence-transformers + scikit-learn + scipy: used later (Stage 3/5) for
#   embedding-based primary metrics and statistical testing; installed now so
#   the environment cell only needs to run once per session.
!pip install -q --no-deps \
    "transformers==4.46.3" \
    "accelerate==1.1.1" \
    "sentencepiece==0.2.0" \
    "pdfplumber==0.11.4" \
    "pymupdf==1.24.13" \
    "huggingface_hub==0.26.2" \
    "sentence-transformers==3.3.1" \
    "scikit-learn==1.5.2" \
    "scipy==1.13.1" \
    "pandas==2.2.2" \
    "rapidfuzz==3.10.1"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
!pip uninstall -y tokenizers -q
!pip install -q --no-deps "tokenizers==0.20.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 42.2 MB/s eta 0:00:00


In [ ]:
!pip install -q --no-deps "pdfminer.six==20231228"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 129.5 MB/s eta 0:00:00


In [ ]:
import transformers, sentence_transformers, sklearn, scipy, pandas, rapidfuzz, fitz, pdfplumber, huggingface_hub
print("all core imports OK")

all core imports OK


In [ ]:
import subprocess
print(subprocess.run(["python", "-c", "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout)

torch 2.11.0+cu128 cuda True



In [ ]:
import torch, torchvision
print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("cuda", torch.version.cuda, "available:", torch.cuda.is_available())

torch 2.11.0+cu128
torchvision 0.26.0+cu128
cuda 12.8 available: True


In [ ]:
# Stage 0.1: Mount Drive and materialize the fixed directory schema.
# Every persistent artifact in this project is written under PROJECT_ROOT so
# that a fresh Colab runtime (local disk wiped) can resume purely by
# re-mounting Drive and re-reading cached files — no recomputation of
# already-completed work.
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = "/content/drive/MyDrive/esg_project"

DIR_SCHEMA = {
    "raw_pdfs_cache":     os.path.join(PROJECT_ROOT, "raw_pdfs_cache"),
    "extracted_text":     os.path.join(PROJECT_ROOT, "extracted_text"),
    "reference_profiles": os.path.join(PROJECT_ROOT, "reference_profiles"),
    "compressed_A":       os.path.join(PROJECT_ROOT, "compressed", "A"),
    "compressed_B":       os.path.join(PROJECT_ROOT, "compressed", "B"),
    "compressed_C":       os.path.join(PROJECT_ROOT, "compressed", "C"),
    "compressed_baseline":os.path.join(PROJECT_ROOT, "compressed", "baseline"),
    "expanded_A":         os.path.join(PROJECT_ROOT, "expanded", "A"),
    "expanded_B":         os.path.join(PROJECT_ROOT, "expanded", "B"),
    "expanded_C":         os.path.join(PROJECT_ROOT, "expanded", "C"),
    "expanded_baseline":  os.path.join(PROJECT_ROOT, "expanded", "baseline"),
    "evaluations":        os.path.join(PROJECT_ROOT, "evaluations"),
    "logs":               os.path.join(PROJECT_ROOT, "logs"),
    "config":             os.path.join(PROJECT_ROOT, "config"),
    "manifest":           os.path.join(PROJECT_ROOT, "manifest"),
    "gen_cache":          os.path.join(PROJECT_ROOT, "gen_cache"),
}

for _name, _path in DIR_SCHEMA.items():
    os.makedirs(_path, exist_ok=True)

print("Directory schema ready under:", PROJECT_ROOT)
for k, v in DIR_SCHEMA.items():
    print(f"  {k:22s} -> {v}")

Mounted at /content/drive
Directory schema ready under: /content/drive/MyDrive/esg_project
  raw_pdfs_cache         -> /content/drive/MyDrive/esg_project/raw_pdfs_cache
  extracted_text         -> /content/drive/MyDrive/esg_project/extracted_text
  reference_profiles     -> /content/drive/MyDrive/esg_project/reference_profiles
  compressed_A           -> /content/drive/MyDrive/esg_project/compressed/A
  compressed_B           -> /content/drive/MyDrive/esg_project/compressed/B
  compressed_C           -> /content/drive/MyDrive/esg_project/compressed/C
  compressed_baseline    -> /content/drive/MyDrive/esg_project/compressed/baseline
  expanded_A             -> /content/drive/MyDrive/esg_project/expanded/A
  expanded_B             -> /content/drive/MyDrive/esg_project/expanded/B
  expanded_C             -> /content/drive/MyDrive/esg_project/expanded/C
  expanded_baseline      -> /content/drive/MyDrive/esg_project/expanded/baseline
  evaluations            -> /content/drive/MyDrive/esg_pr

In [ ]:
# Stage 0.2: Single central config. Every later stage reads exclusively from
# this dict (or the CONFIG object below) — no scattered magic numbers.
# This cell is intentionally the ONLY place experiment-level constants are
# defined; changing an experiment parameter means editing this cell and
# nothing else.
import json
import random
import numpy as np

CONFIG = {
    # --- Reproducibility ---
    "SEED": 42,

    # --- Model (single AI engine for all roles: profile / compression /
    # expansion / LLM-judge). Qwen2.5-7B-Instruct chosen over larger Qwen
    # variants because 4-bit NF4 quantization of the 7B checkpoint fits
    # comfortably in T4's ~15GB VRAM alongside KV-cache headroom for
    # ~10-page-report-length contexts; the 14B variant does not reliably fit
    # with room for generation on Colab Free.
    "MODEL_NAME": "Qwen/Qwen2.5-7B-Instruct",
    "QUANT_CONFIG": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "bfloat16",
        "bnb_4bit_use_double_quant": True,
    },
    "GENERATION_DEFAULTS": {
        "temperature": 0.0,      # deterministic; transformers requires do_sample=False
        "do_sample": False,
        "max_new_tokens": 2048,
        "repetition_penalty": 1.05,
    },

    # --- Dataset acquisition ---
    "HF_DATASET_REPO": "alaaguedda/esg-sustainability-reports",
    "HF_DATASET_REPO_TYPE": "dataset",
    "PDF_SUBDIR": "pdfs",

    # --- Stage 0 filtering / sampling ---
    "PAGE_COUNT_MIN": 10,
    "PAGE_COUNT_MAX": 120,
    "PAGE_COUNT_PREFERRED": 100,
    "MAX_REPORTS": 50,
    "MAX_REPORTS_PER_COMPANY": 3,  # caps over-representation of prolific companies
    "COMPANY_FOLDER_MAX_LEN": 30,
    "COMPANY_FOLDER_REJECT_TERMS": [
        "top", "list", "best", "report", "reports", "sustainability", "esg",
    ],

    # --- Compression ---
    "COMPRESSION_TARGET_RATIO": 0.10,   # fraction of original token count
    "COMPRESSION_TOLERANCE": 0.03,      # +/- absolute tolerance on ratio
    "COMPRESSION_MAX_REGEN_ATTEMPTS": 3,

    # --- Paths (mirrors DIR_SCHEMA; duplicated here so CONFIG is
    # self-contained and serializable independent of the Drive-mount cell) ---
    "PATHS": DIR_SCHEMA,
}

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

# Persist config to Drive so future sessions/conversations can diff against
# it and detect accidental parameter drift.
_config_path = os.path.join(DIR_SCHEMA["config"], "config.json")
with open(_config_path, "w") as f:
    json.dump({k: v for k, v in CONFIG.items()}, f, indent=2, default=str)

print(f"Config written to {_config_path}")
print(json.dumps({k: v for k, v in CONFIG.items() if k != "PATHS"}, indent=2))

Config written to /content/drive/MyDrive/esg_project/config/config.json
{
  "SEED": 42,
  "MODEL_NAME": "Qwen/Qwen2.5-7B-Instruct",
  "QUANT_CONFIG": {
    "load_in_4bit": true,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_use_double_quant": true
  },
  "GENERATION_DEFAULTS": {
    "temperature": 0.0,
    "do_sample": false,
    "max_new_tokens": 2048,
    "repetition_penalty": 1.05
  },
  "HF_DATASET_REPO": "alaaguedda/esg-sustainability-reports",
  "HF_DATASET_REPO_TYPE": "dataset",
  "PDF_SUBDIR": "pdfs",
  "PAGE_COUNT_MIN": 10,
  "PAGE_COUNT_MAX": 120,
  "PAGE_COUNT_PREFERRED": 100,
  "MAX_REPORTS": 50,
  "MAX_REPORTS_PER_COMPANY": 3,
  "COMPANY_FOLDER_MAX_LEN": 30,
  "COMPANY_FOLDER_REJECT_TERMS": [
    "top",
    "list",
    "best",
    "report",
    "reports",
    "sustainability",
    "esg"
  ],
  "COMPRESSION_TARGET_RATIO": 0.1,
  "COMPRESSION_TOLERANCE": 0.03,
  "COMPRESSION_MAX_REGEN_ATTEMPTS": 3
}


In [ ]:
# Stage 0.3: Structured logging. All skip/reject/error events across every
# stage funnel through log_event() so the project has one auditable,
# machine-parseable log rather than scattered print statements. Appends to a
# JSONL file on Drive (survives session wipes) and mirrors to stdout.
import datetime
import threading

_log_lock = threading.Lock()
_LOG_PATH = os.path.join(CONFIG["PATHS"]["logs"], "pipeline_log.jsonl")


def log_event(stage: str, level: str, event: str, **fields):
    """Append a single structured log record.

    Parameters
    ----------
    stage : str
        Pipeline stage identifier, e.g. "stage0_filter", "stage2_compress_A".
    level : str
        One of "INFO", "WARN", "ERROR", "SKIP", "REJECT".
    event : str
        Short machine-stable event name, e.g. "company_rejected".
    **fields :
        Arbitrary JSON-serializable context (reasons, ids, metrics, etc).
    """
    record = {
        "ts": datetime.datetime.now(datetime.UTC).isoformat(),
        "stage": stage,
        "level": level,
        "event": event,
        **fields,
    }
    line = json.dumps(record, default=str)
    with _log_lock:
        with open(_LOG_PATH, "a") as f:
            f.write(line + "\n")
    print(f"[{record['ts']}] [{level}] [{stage}] {event} :: "
          f"{ {k: v for k, v in fields.items()} }")
    return record


log_event("bootstrap", "INFO", "logging_initialized", log_path=_LOG_PATH)

[2026-07-05T09:16:10.922381+00:00] [INFO] [bootstrap] logging_initialized :: {'log_path': '/content/drive/MyDrive/esg_project/logs/pipeline_log.jsonl'}


{'ts': '2026-07-05T09:16:10.922381+00:00',
 'stage': 'bootstrap',
 'level': 'INFO',
 'event': 'logging_initialized',
 'log_path': '/content/drive/MyDrive/esg_project/logs/pipeline_log.jsonl'}

In [ ]:
# Stage 0.4: Generic content-addressed cache for expensive steps (extraction,
# profile generation, compression, expansion, evaluation, and raw LLM calls).
# Keyed by a hash of the *semantic inputs* of the call (not just a filename),
# so identical inputs always resolve to the same cache entry even across
# stages/conversations, and changed prompts/config automatically invalidate
# stale cache entries instead of silently reusing them.
import hashlib


def make_cache_key(*parts) -> str:
    """Deterministic hash over an arbitrary sequence of stringifiable parts.

    Every caller passes the exact set of inputs that determine the output
    (e.g. text content, prompt template id, model name, target ratio) so
    that a change to any of them naturally produces a new key.
    """
    h = hashlib.sha256()
    for p in parts:
        h.update(str(p).encode("utf-8"))
        h.update(b"\x1f")  # unit separator, avoids trivial collisions across parts
    return h.hexdigest()[:24]


def cache_path(cache_dir: str, key: str, ext: str = "json") -> str:
    return os.path.join(cache_dir, f"{key}.{ext}")


def cache_get_json(cache_dir: str, key: str):
    """Return cached JSON object, or None if not present / unreadable."""
    p = cache_path(cache_dir, key, "json")
    if not os.path.exists(p):
        return None
    try:
        with open(p, "r") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        log_event("cache", "WARN", "cache_read_failed", path=p)
        return None


def cache_set_json(cache_dir: str, key: str, obj) -> str:
    """Write obj as JSON to the cache atomically (write-then-rename) so a
    Colab interruption mid-write never leaves a corrupt cache entry that
    would silently poison a later resume."""
    os.makedirs(cache_dir, exist_ok=True)
    p = cache_path(cache_dir, key, "json")
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, p)
    return p


def cache_get_text(cache_dir: str, key: str):
    p = cache_path(cache_dir, key, "txt")
    if not os.path.exists(p):
        return None
    with open(p, "r", encoding="utf-8") as f:
        return f.read()


def cache_set_text(cache_dir: str, key: str, text: str) -> str:
    os.makedirs(cache_dir, exist_ok=True)
    p = cache_path(cache_dir, key, "txt")
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, p)
    return p


log_event("bootstrap", "INFO", "cache_utils_ready", gen_cache_dir=CONFIG["PATHS"]["gen_cache"])

[2026-07-05T09:16:13.810014+00:00] [INFO] [bootstrap] cache_utils_ready :: {'gen_cache_dir': '/content/drive/MyDrive/esg_project/gen_cache'}


{'ts': '2026-07-05T09:16:13.810014+00:00',
 'stage': 'bootstrap',
 'level': 'INFO',
 'event': 'cache_utils_ready',
 'gen_cache_dir': '/content/drive/MyDrive/esg_project/gen_cache'}

In [ ]:
# Stage 0.5: Load the single AI engine once per session, reused by every
# stage (profile generation, compression, expansion, LLM-judge evaluation).
# Deterministic generation (do_sample=False, greedy) + fixed seed satisfy the
# project's determinism requirement; note that "temperature=0" for an
# autoregressive HF model is implemented as greedy decoding (do_sample=False)
# since temperature itself is only meaningful under sampling.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import set_seed as hf_set_seed

hf_set_seed(CONFIG["SEED"])

_MODEL = None
_TOKENIZER = None


def get_model_and_tokenizer():
    """Lazily load and cache the model/tokenizer as module-level singletons
    so repeated calls across Stage 1-6 cells do not reload the 7B checkpoint.
    """
    global _MODEL, _TOKENIZER
    if _MODEL is not None:
        return _MODEL, _TOKENIZER

    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=CONFIG["QUANT_CONFIG"]["load_in_4bit"],
        bnb_4bit_quant_type=CONFIG["QUANT_CONFIG"]["bnb_4bit_quant_type"],
        bnb_4bit_compute_dtype=getattr(torch, CONFIG["QUANT_CONFIG"]["bnb_4bit_compute_dtype"]),
        bnb_4bit_use_double_quant=CONFIG["QUANT_CONFIG"]["bnb_4bit_use_double_quant"],
    )

    log_event("model_load", "INFO", "loading_model", model=CONFIG["MODEL_NAME"])

    _TOKENIZER = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
    _MODEL = AutoModelForCausalLM.from_pretrained(
        CONFIG["MODEL_NAME"],
        quantization_config=quant_cfg,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    _MODEL.eval()

    log_event("model_load", "INFO", "model_ready",
              device=str(_MODEL.device) if hasattr(_MODEL, "device") else "sharded")
    return _MODEL, _TOKENIZER


def generate(prompt: str, system: str = None, max_new_tokens: int = None,
             cache_dir: str = None, cache_extra: str = "") -> str:
    gen_kwargs = dict(CONFIG["GENERATION_DEFAULTS"])
    if max_new_tokens is not None:
        gen_kwargs["max_new_tokens"] = max_new_tokens

    key = None
    if cache_dir is not None:
        key = make_cache_key(CONFIG["MODEL_NAME"], system or "", prompt,
                              json.dumps(gen_kwargs, sort_keys=True), cache_extra)
        cached = cache_get_text(cache_dir, key)
        if cached is not None:
            return cached

    model, tok = get_model_and_tokenizer()
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    input_ids = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            input_ids,
            do_sample=gen_kwargs["do_sample"],
            max_new_tokens=gen_kwargs["max_new_tokens"],
            repetition_penalty=gen_kwargs["repetition_penalty"],
            pad_token_id=tok.eos_token_id,
        )
    new_tokens = out_ids[0][input_ids.shape[1]:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()

    # explicit cleanup — free large tensors immediately instead of waiting
    # for Python's garbage collector, so the allocator can reclaim/reuse
    # the memory before the next generate() call
    del input_ids, out_ids, new_tokens
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    if cache_dir is not None:
        cache_set_text(cache_dir, key, text)

    return text


def count_tokens(text: str) -> int:
    """Canonical token-counting function — used everywhere a report's size
    or a compression ratio needs to be measured, so every stage agrees on
    what a 'token' is (the target model's own tokenizer, not a proxy like
    whitespace splitting)."""
    _, tok = get_model_and_tokenizer()
    return len(tok(text, add_special_tokens=False)["input_ids"])

In [ ]:
# Stage 0.6: List and filter the HF dataset's company folders WITHOUT
# downloading any PDFs yet. This is a metadata-only pass so filtering is
# cheap and fully re-runnable; only folders/files surviving the filter are
# ever downloaded (Stage 0.7+).
from huggingface_hub import HfApi
import re

_api = HfApi()


def list_pdf_repo_files():
    """Return all repo-relative paths under pdfs/ in the HF dataset repo."""
    all_files = _api.list_repo_files(
        repo_id=CONFIG["HF_DATASET_REPO"],
        repo_type=CONFIG["HF_DATASET_REPO_TYPE"],
    )
    prefix = CONFIG["PDF_SUBDIR"] + "/"
    return [f for f in all_files if f.startswith(prefix) and f.lower().endswith(".pdf")]


def is_valid_company_folder(folder: str) -> (bool, str):
    """Apply the explicit company-folder heuristic (all rules must pass).
    Returns (is_valid, reason) so every decision is logged, not just failures.
    """
    if not folder:
        return False, "empty_folder_name"
    if not folder.isalpha():
        # single alphabetic token: letters only, no digits/underscores/spaces
        return False, "not_single_alphabetic_token"
    if len(folder) > CONFIG["COMPANY_FOLDER_MAX_LEN"]:
        return False, f"exceeds_max_len_{CONFIG['COMPANY_FOLDER_MAX_LEN']}"
    lower = folder.lower()
    for term in CONFIG["COMPANY_FOLDER_REJECT_TERMS"]:
        if term in lower:
            return False, f"contains_reject_term:{term}"
    return True, "ok"


def build_filtered_file_index():
    """Group repo PDF paths by company folder and apply the folder filter.
    Every rejected folder is logged exactly once with its reason.
    Returns dict: company -> list of repo-relative pdf paths.
    """
    raw_files = list_pdf_repo_files()
    log_event("stage0_filter", "INFO", "repo_pdf_listing_complete", count=len(raw_files))

    by_company = {}
    for path in raw_files:
        # expected shape: pdfs/<company>/<filename>.pdf
        parts = path.split("/")
        if len(parts) != 3:
            log_event("stage0_filter", "SKIP", "unexpected_path_shape", path=path)
            continue
        _, company, filename = parts
        by_company.setdefault(company, []).append(path)

    seen_folders = set()
    filtered = {}
    for company, paths in by_company.items():
        if company in seen_folders:
            continue
        seen_folders.add(company)
        ok, reason = is_valid_company_folder(company)
        if ok:
            filtered[company] = paths
        else:
            log_event("stage0_filter", "REJECT", "company_folder_rejected",
                       folder=company, reason=reason, n_files=len(paths))

    log_event("stage0_filter", "INFO", "folder_filter_complete",
              accepted=len(filtered), rejected=len(seen_folders) - len(filtered))
    return filtered


FILTERED_FILE_INDEX = build_filtered_file_index()
print(f"Accepted companies: {len(FILTERED_FILE_INDEX)}")

[2026-07-05T09:16:26.777905+00:00] [INFO] [stage0_filter] repo_pdf_listing_complete :: {'count': 2331}
[2026-07-05T09:16:26.802001+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '10_mining_startups_to_watch_in_2026_startus_insights', 'reason': 'not_single_alphabetic_token', 'n_files': 7}
[2026-07-05T09:16:26.834446+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '10_sustainable_logistics_companies_[2026]_startus_insights', 'reason': 'not_single_alphabetic_token', 'n_files': 10}
[2026-07-05T09:16:26.841521+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '2024_esg_082125_v26_cw_-_intuitive.com', 'reason': 'not_single_alphabetic_token', 'n_files': 1}
[2026-07-05T09:16:26.845150+00:00] [REJECT] [stage0_filter] company_folder_rejected :: {'folder': '20_unique_transportation_startups_you_should_know_in_2025', 'reason': 'not_single_alphabetic_token', 'n_files': 1}
[2026-07-05T09:16:26.847939+00:00] [REJECT] [stage0_filter] com

In [ ]:
# Stage 0.7: Download candidate PDFs (only from accepted companies),
# check page count locally, and select up to MAX_REPORTS reports meeting the
# 10-15 page window (preferring ~10 pages), capped per company, with a fixed
# seed for the random component. Downloaded PDFs are cached to
# raw_pdfs_cache/ on Drive so re-running after an interruption does not
# re-download files already fetched.
import fitz  # pymupdf, used here only for fast page counts
from huggingface_hub import hf_hub_download

MANIFEST_PATH = os.path.join(CONFIG["PATHS"]["manifest"], "stage0_manifest.json")


def _local_pdf_cache_path(repo_path: str) -> str:
    safe_name = repo_path.replace("/", "__")
    return os.path.join(CONFIG["PATHS"]["raw_pdfs_cache"], safe_name)


def ensure_pdf_downloaded(repo_path: str) -> str:
    """Download a single PDF from the HF dataset into the Drive-backed raw
    cache if not already present; return the local path either way."""
    local_path = _local_pdf_cache_path(repo_path)
    if os.path.exists(local_path):
        return local_path
    downloaded = hf_hub_download(
        repo_id=CONFIG["HF_DATASET_REPO"],
        repo_type=CONFIG["HF_DATASET_REPO_TYPE"],
        filename=repo_path,
    )
    with open(downloaded, "rb") as src, open(local_path, "wb") as dst:
        dst.write(src.read())
    return local_path


def get_page_count(local_pdf_path: str):
    try:
        with fitz.open(local_pdf_path) as doc:
            return doc.page_count
    except Exception as e:
        log_event("stage0_select", "ERROR", "pdf_open_failed",
                   path=local_pdf_path, error=str(e))
        return None


def select_reports():
    """Build the Stage 0 manifest: up to MAX_REPORTS reports in the target
    page-count window, capped per company, sampled with a fixed seed among
    qualifying reports when more candidates qualify than needed.

    Resumable: if a manifest already exists on Drive, it is loaded and
    returned as-is rather than recomputed (Stage 0 selection is treated as
    locked once produced, matching the "reference profile is fixed ground
    truth" philosophy applied one stage earlier).
    """
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            manifest = json.load(f)
        log_event("stage0_select", "INFO", "manifest_loaded_from_cache",
                   n_reports=len(manifest["reports"]))
        return manifest

    rng = random.Random(CONFIG["SEED"])
    companies = sorted(FILTERED_FILE_INDEX.keys())  # sort for determinism before shuffle
    rng.shuffle(companies)

    qualifying = []  # list of dicts, evaluated lazily company by company
    per_company_count = {}

    for company in companies:
        if len(qualifying) >= CONFIG["MAX_REPORTS"] * 3:
            # enough candidates gathered to make a well-supported final
            # selection without scanning the entire (large) repo
            break
        paths = list(FILTERED_FILE_INDEX[company])
        rng.shuffle(paths)
        for repo_path in paths:
            if per_company_count.get(company, 0) >= CONFIG["MAX_REPORTS_PER_COMPANY"]:
                break
            local_path = ensure_pdf_downloaded(repo_path)
            n_pages = get_page_count(local_path)
            if n_pages is None:
                continue
            if not (CONFIG["PAGE_COUNT_MIN"] <= n_pages <= CONFIG["PAGE_COUNT_MAX"]):
                log_event("stage0_select", "SKIP", "page_count_out_of_range",
                           repo_path=repo_path, n_pages=n_pages)
                continue
            qualifying.append({
                "company": company,
                "repo_path": repo_path,
                "local_pdf_path": local_path,
                "n_pages": n_pages,
                "dist_from_preferred": abs(n_pages - CONFIG["PAGE_COUNT_PREFERRED"]),
            })
            per_company_count[company] = per_company_count.get(company, 0) + 1

    log_event("stage0_select", "INFO", "qualifying_candidates_found", n=len(qualifying))

    # Selection method: prefer reports closest to the preferred page count;
    # break ties with the fixed-seed shuffle order already applied above.
    # This is a deterministic, documented sampling method as required.
    qualifying.sort(key=lambda r: r["dist_from_preferred"])
    selected = qualifying[:CONFIG["MAX_REPORTS"]]

    for r in selected:
        r["report_id"] = make_cache_key(r["company"], r["repo_path"])

    manifest = {
        "seed": CONFIG["SEED"],
        "sampling_method": (
            "shuffle companies (fixed seed) -> per-company cap "
            f"({CONFIG['MAX_REPORTS_PER_COMPANY']}) -> filter by page range "
            f"[{CONFIG['PAGE_COUNT_MIN']},{CONFIG['PAGE_COUNT_MAX']}] -> "
            f"rank by |pages - {CONFIG['PAGE_COUNT_PREFERRED']}| -> "
            f"take top {CONFIG['MAX_REPORTS']}"
        ),
        "n_qualifying_total": len(qualifying),
        "n_selected": len(selected),
        "target_max_reports": CONFIG["MAX_REPORTS"],
        "reports": selected,
    }

    if len(selected) < CONFIG["MAX_REPORTS"]:
        log_event("stage0_select", "WARN", "fewer_than_target_reports",
                   found=len(selected), target=CONFIG["MAX_REPORTS"])

    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    log_event("stage0_select", "INFO", "manifest_written",
              path=MANIFEST_PATH, n_reports=len(selected))
    return manifest


STAGE0_MANIFEST = select_reports()
print(f"Selected {STAGE0_MANIFEST['n_selected']} reports "
      f"(of {STAGE0_MANIFEST['n_qualifying_total']} qualifying candidates).")

[2026-07-05T09:16:40.520915+00:00] [INFO] [stage0_select] manifest_loaded_from_cache :: {'n_reports': 50}
Selected 50 reports (of 118 qualifying candidates).


In [ ]:
# @title
# Stage 0.8: Text extraction for every selected report, cached to Drive keyed
# by report_id (company + filename hash from the manifest) so re-extraction
# is skipped on resume. Uses pdfplumber for layout-aware extraction.
import pdfplumber


def extract_text_from_pdf(local_pdf_path: str) -> str:
    pages_text = []
    with pdfplumber.open(local_pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text() or ""
            pages_text.append(t)
    return "\n\n".join(pages_text).strip()


def extracted_text_path(report_id: str) -> str:
    return os.path.join(CONFIG["PATHS"]["extracted_text"], f"{report_id}.txt")


def get_or_extract_text(report: dict) -> str:
    path = extracted_text_path(report["report_id"])
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    text = extract_text_from_pdf(report["local_pdf_path"])
    if not text:
        log_event("stage0_extract", "WARN", "empty_extraction",
                   report_id=report["report_id"], repo_path=report["repo_path"])
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)
    return text


def run_stage0_extraction(manifest: dict) -> dict:
    """Extract (or load cached) text for every report in the manifest;
    attach n_tokens (canonical tokenizer-based count) to each record and
    persist the enriched manifest so downstream stages have token counts
    available without recomputation."""
    enriched = []
    for report in manifest["reports"]:
        text = get_or_extract_text(report)
        n_tokens = count_tokens(text) if text else 0
        enriched_report = dict(report)
        enriched_report["extracted_text_path"] = extracted_text_path(report["report_id"])
        enriched_report["n_chars"] = len(text)
        enriched_report["n_tokens_original"] = n_tokens
        enriched.append(enriched_report)
        log_event("stage0_extract", "INFO", "extraction_ready",
                   report_id=report["report_id"], n_pages=report["n_pages"],
                   n_tokens=n_tokens)

    manifest["reports"] = enriched
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    return manifest


STAGE0_MANIFEST = run_stage0_extraction(STAGE0_MANIFEST)

import pandas as pd
_manifest_df = pd.DataFrame(STAGE0_MANIFEST["reports"])
_manifest_csv_path = os.path.join(CONFIG["PATHS"]["manifest"], "stage0_manifest.csv")
_manifest_df.to_csv(_manifest_csv_path, index=False)
print(f"Stage 0 complete: {len(_manifest_df)} reports extracted.")
print(f"Manifest CSV: {_manifest_csv_path}")
_manifest_df[["company", "n_pages", "n_tokens_original"]].describe(include="all")

[2026-07-05T09:16:52.944760+00:00] [INFO] [model_load] loading_model :: {'model': 'Qwen/Qwen2.5-7B-Instruct'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[2026-07-05T09:21:16.912290+00:00] [INFO] [model_load] model_ready :: {'device': 'cuda:0'}
[2026-07-05T09:21:16.943321+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': 'bfd1b3b0ebe8410f3424243f', 'n_pages': 10, 'n_tokens': 3246}
[2026-07-05T09:21:17.878870+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '852815f0465c6a55e522d6cf', 'n_pages': 10, 'n_tokens': 8928}
[2026-07-05T09:21:18.816210+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '5e6d98d4006f9eec322846dd', 'n_pages': 10, 'n_tokens': 3485}
[2026-07-05T09:21:19.634699+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': 'bf96fa1c7ff813f490e79b07', 'n_pages': 10, 'n_tokens': 2631}
[2026-07-05T09:21:20.495296+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '35ed2dd1c7889989991747ee', 'n_pages': 10, 'n_tokens': 4228}
[2026-07-05T09:21:21.539481+00:00] [INFO] [stage0_extract] extraction_ready :: {'report_id': '4772a1d980f05b15994e318d', 'n_pages': 11, 'n_

,company,n_pages,n_tokens_original
count,50,50.00000,50.000000
unique,33,NaN,NaN
top,cropx,NaN,NaN
freq,3,NaN,NaN
mean,NaN,19.16000,8459.180000
std,NaN,7.12386,6243.121935
min,NaN,10.00000,1710.000000
25%,NaN,12.25000,4225.750000
50%,NaN,17.50000,6547.500000
75%,NaN,24.75000,10983.750000


In [ ]:
# Stage 1.0 v2: Flattened Reference Profile schema. v1 (nested per-item
# objects for kpis/financial_metrics/environmental_metrics/targets/
# dates_events/numerical_values) is DEPRECATED but NOT deleted — profiles
# already generated under v1 remain valid ground truth and are read via the
# version-aware adapter (next cell), not regenerated. v2 is used for all
# NEW profile generation going forward.
#
# Change vs v1: every field is now `array of string` (or plain string).
# Structured facts that used to be sub-objects are now single, densely
# formatted strings, e.g.:
#   kpi:               "Scope 1 emissions: 98,400 tCO2e (FY2024)"
#   financial_metric:  "Revenue: 4.2B USD"
#   environmental_metric: "Water withdrawal: 1.1M m3"
#   target:            "Net zero by 2040 (target: net zero, year: 2040)"
#   date_event:        "2024-03-01: Published annual sustainability report"
#   numerical_value:   "18% (context: YoY Scope 1 reduction)"
# This removes the nested-object formatting burden that was the primary
# driver of empty arrays (see conversation analysis), while keeping every
# fact machine-parseable downstream via regex/fuzzy matching (Stage 3/5/6
# metrics already fuzzy-match on substrings, so a flat "name: value unit"
# string is equally usable as a matching target — often MORE robust, since
# it removes brittle exact key-based lookups).
REFERENCE_PROFILE_SCHEMA_VERSION_V2 = "2.0"

REFERENCE_PROFILE_SCHEMA_V2 = {
    "schema_version": REFERENCE_PROFILE_SCHEMA_VERSION_V2,
    "type": "object",
    "required": [
        "company", "year", "industry", "industry_inference_method",
        "executive_summary", "esg_topics", "entities", "organizations",
        "stakeholders", "numerical_values", "kpis", "financial_metrics",
        "environmental_metrics", "governance_information", "social_information",
        "risks", "opportunities", "commitments", "targets", "future_plans",
        "regulations_frameworks_standards", "dates_events",
    ],
    "properties": {
        "company": {"type": "string"},
        "year": {"type": ["string", "null"]},
        "industry": {"type": "string"},
        "industry_inference_method": {"type": "string"},
        "executive_summary": {"type": "string"},
        # Every field below: flat array of pre-formatted strings.
        **{
            f: {"type": "array", "items": {"type": "string"}}
            for f in [
                "esg_topics", "entities", "organizations", "stakeholders",
                "numerical_values", "kpis", "financial_metrics",
                "environmental_metrics", "governance_information",
                "social_information", "risks", "opportunities", "commitments",
                "targets", "future_plans", "regulations_frameworks_standards",
                "dates_events",
            ]
        },
    },
}

# Keep the v1 schema object (from the original Stage 1.0 cell) resident
# under its own name so the adapter cell below can still validate/read
# already-generated v1 profiles by version, rather than silently guessing
# their shape.
#REFERENCE_PROFILE_SCHEMA_V1 = REFERENCE_PROFILE_SCHEMA  # alias to the original
                                                          # nested schema object

# Going forward, REFERENCE_PROFILE_SCHEMA / REFERENCE_PROFILE_SCHEMA_VERSION
# point at v2 — every NEW generation call uses this.
REFERENCE_PROFILE_SCHEMA = REFERENCE_PROFILE_SCHEMA_V2
REFERENCE_PROFILE_SCHEMA_VERSION = REFERENCE_PROFILE_SCHEMA_VERSION_V2

_schema_v2_path = os.path.join(CONFIG["PATHS"]["config"], "reference_profile_schema_v2.json")
with open(_schema_v2_path, "w") as f:
    json.dump(REFERENCE_PROFILE_SCHEMA_V2, f, indent=2)
log_event("stage1_schema", "INFO", "schema_v2_locked_and_written",
          version=REFERENCE_PROFILE_SCHEMA_VERSION_V2, path=_schema_v2_path)

[2026-07-05T09:22:05.877405+00:00] [INFO] [stage1_schema] schema_v2_locked_and_written :: {'version': '2.0', 'path': '/content/drive/MyDrive/esg_project/config/reference_profile_schema_v2.json'}


{'ts': '2026-07-05T09:22:05.877405+00:00',
 'stage': 'stage1_schema',
 'level': 'INFO',
 'event': 'schema_v2_locked_and_written',
 'version': '2.0',
 'path': '/content/drive/MyDrive/esg_project/config/reference_profile_schema_v2.json'}

In [ ]:
# Stage 1.0b: Sample a fixed subset of 5 reports from the Stage 0 manifest
# for this study run, instead of processing all 50. Deterministic (fixed
# seed) and cached to disk so re-running the notebook reuses the same 5
# reports rather than resampling.
STAGE1_SAMPLE_PATH = os.path.join(CONFIG["PATHS"]["manifest"], "stage1_sample.json")
N_SAMPLE = 5

def get_stage1_sample(manifest: dict, n: int = N_SAMPLE) -> dict:
    if os.path.exists(STAGE1_SAMPLE_PATH):
        with open(STAGE1_SAMPLE_PATH, "r") as f:
            sample = json.load(f)
        log_event("stage1_sample", "INFO", "sample_loaded_from_cache",
                   n_reports=len(sample["reports"]))
        return sample

    rng = random.Random(CONFIG["SEED"])
    reports = list(manifest["reports"])
    rng.shuffle(reports)
    selected = reports[:n]

    sample = {"seed": CONFIG["SEED"], "n_sample": len(selected), "reports": selected}
    with open(STAGE1_SAMPLE_PATH, "w") as f:
        json.dump(sample, f, indent=2, default=str)
    log_event("stage1_sample", "INFO", "sample_written",
               path=STAGE1_SAMPLE_PATH, n_reports=len(selected))
    return sample


STAGE1_SAMPLE = get_stage1_sample(STAGE0_MANIFEST, N_SAMPLE)
print(f"Sampled {STAGE1_SAMPLE['n_sample']} reports for Stage 1: "
      f"{[r['report_id'] for r in STAGE1_SAMPLE['reports']]}")

[2026-07-05T09:48:50.869529+00:00] [INFO] [stage1_sample] sample_loaded_from_cache :: {'n_reports': 5}
Sampled 5 reports for Stage 1: ['a9a45ce8faeb53d10d8d0dd3', '8777dcf4b50e5d00da6e73de', '26aef234bc0eb9665b42be0d', '15f5ef3cd925ac417b95388c', '35ed2dd1c7889989991747ee']


In [ ]:
# check report sizes in your sample before generating
for r in STAGE1_SAMPLE["reports"]:
    print(r["report_id"], r.get("n_tokens_original"))

a9a45ce8faeb53d10d8d0dd3 5674
8777dcf4b50e5d00da6e73de 7770
26aef234bc0eb9665b42be0d 8162
15f5ef3cd925ac417b95388c 6496
35ed2dd1c7889989991747ee 4228


In [ ]:
print(f"Model memory footprint: {_MODEL.get_memory_footprint() / 1e9:.2f} GB")

# check if any layers are actually 4-bit
import bitsandbytes as bnb
n_4bit, n_other = 0, 0
for name, module in _MODEL.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        n_4bit += 1
    elif isinstance(module, torch.nn.Linear):
        n_other += 1
print("Linear4bit layers:", n_4bit, "| plain Linear layers:", n_other)

Model memory footprint: 5.44 GB
Linear4bit layers: 196 | plain Linear layers: 1


In [ ]:
print(_MODEL.hf_device_map)
print(_MODEL.config.quantization_config)

{'': 0}
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(torch.cuda.memory_allocated()/1e9, "GB allocated")
print(torch.cuda.memory_reserved()/1e9, "GB reserved")

5.97705728 GB allocated
6.09222656 GB reserved


In [ ]:
# Stage 1.1 v2: Reference Profile generation — flat schema, greedier prompt,
# coerce-before-validate (fixes type mismatches locally so the RETRY LOOP is
# reserved for genuine JSON-parse failures, not trivial type coercion —
# eliminating the wasted regeneration calls seen in the logs), and a
# version-aware read adapter so v1 (nested) profiles already on Drive keep
# working in every downstream stage without regeneration.
import gc
import jsonschema
try:
    import jsonschema  # noqa: F811
except ImportError:
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "jsonschema==4.23.0"])
    import jsonschema  # noqa: F811

REFERENCE_PROFILE_PROMPT_VERSION = "v2"
MAX_PROFILE_INPUT_TOKENS = 4000
PROFILE_MAX_NEW_TOKENS = 3072

# Rebalanced system prompt: explicit permission (even instruction) to be
# inclusive/generous, paired with a SINGLE narrow anti-hallucination
# constraint (don't invent facts that aren't in the text at all) instead of
# many overlapping "be exact" phrasings that previously stacked up into an
# overall "when unsure, omit" bias.
_PROFILE_SYSTEM_PROMPT = (
    "You are an ESG/sustainability report analyst. Your job is to extract "
    "AS MUCH relevant information as you can find in the report into the "
    "given JSON fields. Be GENEROUS and INCLUSIVE: if a fact is present in "
    "the text and plausibly belongs in a field, PUT IT THERE even if you "
    "are not 100% sure it is the perfect category — a reasonable categorization "
    "is far more useful than leaving a field empty. The only hard rule is: "
    "do not invent facts, numbers, or names that are not present anywhere in "
    "the source text. Missing a real fact from the report is a WORSE mistake "
    "than placing a real fact in a slightly imperfect field. "
    "Output ONLY a single valid JSON object matching the schema. No prose, "
    "no markdown code fences, no commentary before or after the JSON."
)

# One compact positive example demonstrating DENSITY (several items per
# array, flat pre-formatted strings) rather than only describing the format
# in prose — small instruction-tuned models imitate demonstrated density
# far more reliably than they follow abstract "be thorough" instructions.
_PROFILE_FEW_SHOT_EXAMPLE = """EXAMPLE (abbreviated report -> abbreviated profile):
REPORT EXCERPT: "GreenCorp cut Scope 1 emissions 18% YoY (120,000->98,400 tCO2e,
2023-2024) via renewable electricity at its 3 largest plants. The Audit
Committee met 4 times in FY2024 and launched a whistleblower hotline (42
reports, 12 substantiated). GreenCorp targets net zero by 2040 and follows
the GRI Standards."

EXAMPLE PROFILE (partial):
{
  "esg_topics": ["emissions reduction", "renewable electricity", "whistleblower protection"],
  "kpis": ["Scope 1 emissions: 98,400 tCO2e (2024)", "Scope 1 emissions: 120,000 tCO2e (2023)"],
  "governance_information": ["Audit Committee met 4 times in FY2024", "Whistleblower hotline: 42 reports, 12 substantiated"],
  "targets": ["Net zero by 2040"],
  "regulations_frameworks_standards": ["GRI Standards"]
}
Notice EVERY concrete fact in the excerpt appears somewhere above — that is the density expected from you."""


def _truncate_for_profile_input(report_text: str, max_tokens: int = MAX_PROFILE_INPUT_TOKENS):
    _, tok = get_model_and_tokenizer()
    ids = tok(report_text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return report_text, False
    truncated_ids = ids[:max_tokens]
    return tok.decode(truncated_ids, skip_special_tokens=True), True


def _build_profile_prompt(report_text: str) -> str:
    schema_keys = ", ".join(REFERENCE_PROFILE_SCHEMA["required"])
    return f"""{_PROFILE_FEW_SHOT_EXAMPLE}

Now extract a Reference Profile from the REAL report below.

Return a single JSON object with EXACTLY these top-level keys (no more, no fewer):
{schema_keys}

Format rules:
- "company", "year", "industry", "industry_inference_method", "executive_summary": plain strings.
- ALL other fields are arrays of plain strings (no nested objects, no sub-keys).
- For numeric/structured facts, pack the detail into ONE readable string, e.g.:
  kpis: "Scope 1 emissions: 98,400 tCO2e (FY2024)"
  financial_metrics: "Revenue: 4.2B USD (FY2024)"
  environmental_metrics: "Water withdrawal: 1.1M m3"
  targets: "Net zero by 2040"
  dates_events: "2024-03-01: Published annual sustainability report"
  numerical_values: "18% YoY reduction in Scope 1 emissions"
- List EVERY distinct instance you find, deduplicated. Aim for completeness over caution.
- "industry_inference_method": one short phrase, e.g. "stated explicitly" or
  "inferred from described products/services".
- Do not invent facts absent from the text below, but DO include every real fact
  you find, even if you must place it in the closest-fitting field.

REPORT TEXT:
\"\"\"
{report_text}
\"\"\"

Return ONLY the JSON object."""


def _extract_json_object(raw_text: str) -> dict:
    text = raw_text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    start = text.find("{")
    if start == -1:
        raise ValueError("no_json_object_found")
    candidate = text[start:]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Truncated output recovery: close any unbalanced brackets/braces and
    # strip a dangling trailing comma/partial value, then retry once.
    # This turns "generation hit max_new_tokens mid-array" into a usable
    # partial profile instead of a hard failure.
    depth_curly, depth_square = 0, 0
    in_string, escape = False, False
    last_safe_idx = 0
    for i, ch in enumerate(candidate):
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth_curly += 1
        elif ch == "}":
            depth_curly -= 1
        elif ch == "[":
            depth_square += 1
        elif ch == "]":
            depth_square -= 1
        if ch in ",}]" and depth_curly >= 0:
            last_safe_idx = i + 1

    repaired = candidate[:last_safe_idx].rstrip().rstrip(",")
    repaired += "]" * max(depth_square, 0) + "}" * max(depth_curly, 0)
    return json.loads(repaired)


def profile_path(report_id: str) -> str:
    return os.path.join(CONFIG["PATHS"]["reference_profiles"], f"{report_id}.json")


def _free_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _coerce_to_schema_v2(obj: dict) -> dict:
    """Fix common type mismatches locally (no model call) BEFORE validation,
    so schema-validation retries are reserved for genuinely unparsable
    output, not trivial coercions like None->"" or int year->str. This is
    exactly the class of retry seen in the logs (`None is not of type
    'string'`, `2022 is not of type 'string'`) — both fixed here for free.
    """
    scalar_string_fields = ["company", "industry", "industry_inference_method",
                             "executive_summary"]
    for f in scalar_string_fields:
        v = obj.get(f)
        if v is None:
            obj[f] = ""
        elif not isinstance(v, str):
            obj[f] = str(v)

    y = obj.get("year")
    if y is not None and not isinstance(y, str):
        obj["year"] = str(y)

    flat_array_fields = [
        "esg_topics", "entities", "organizations", "stakeholders",
        "numerical_values", "kpis", "financial_metrics", "environmental_metrics",
        "governance_information", "social_information", "risks", "opportunities",
        "commitments", "targets", "future_plans", "regulations_frameworks_standards",
        "dates_events",
    ]
    for f in flat_array_fields:
        items = obj.get(f, [])
        if not isinstance(items, list):
            items = [items] if items else []
        fixed = []
        for item in items:
            if isinstance(item, str):
                if item.strip():
                    fixed.append(item.strip())
            elif isinstance(item, dict):
                # Legacy nested-object-shaped item slipped through despite
                # the flat prompt: flatten it into one readable string
                # instead of discarding the fact it represents.
                parts = [f"{k}: {v}" for k, v in item.items() if v not in (None, "")]
                if parts:
                    fixed.append(", ".join(parts))
            elif item not in (None, ""):
                fixed.append(str(item))
        obj[f] = fixed

    # Drop any unexpected extra keys the model may have added, and ensure
    # every required key is present at all (missing key, not just wrong
    # type, still needs to raise so the retry loop catches truly broken
    # output rather than silently proceeding with a partial profile).
    return obj


def generate_reference_profile(report: dict) -> dict:
    out_path = profile_path(report["report_id"])
    if os.path.exists(out_path):
        with open(out_path, "r") as f:
            return json.load(f)

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        report_text = f.read()

    report_text, was_truncated = _truncate_for_profile_input(report_text)
    if was_truncated:
        log_event("stage1_profile", "WARN", "input_truncated_for_memory",
                   report_id=report["report_id"], max_tokens=MAX_PROFILE_INPUT_TOKENS)

    prompt = _build_profile_prompt(report_text)
    raw = generate(
        prompt,
        system=_PROFILE_SYSTEM_PROMPT,
        max_new_tokens=PROFILE_MAX_NEW_TOKENS,
        cache_dir=CONFIG["PATHS"]["gen_cache"],
        cache_extra=f"profile:{REFERENCE_PROFILE_PROMPT_VERSION}:single",
    )

    obj = None
    try:
        parsed = _extract_json_object(raw)
        for k in REFERENCE_PROFILE_SCHEMA["required"]:
            parsed.setdefault(k, "" if k in
                ("company", "year", "industry", "industry_inference_method",
                 "executive_summary") else [])
        parsed = _coerce_to_schema_v2(parsed)
        jsonschema.validate(instance=parsed, schema=REFERENCE_PROFILE_SCHEMA)
        parsed["_meta"] = {
            "report_id": report["report_id"],
            "schema_version": REFERENCE_PROFILE_SCHEMA_VERSION,
            "prompt_version": REFERENCE_PROFILE_PROMPT_VERSION,
            "input_truncated": was_truncated,
        }
        tmp = out_path + ".tmp"
        with open(tmp, "w") as f:
            json.dump(parsed, f, indent=2, default=str)
        os.replace(tmp, out_path)
        log_event("stage1_profile", "INFO", "profile_generated", report_id=report["report_id"])
        obj = parsed
    except (json.JSONDecodeError, ValueError, jsonschema.ValidationError) as e:
        log_event("stage1_profile", "ERROR", "profile_generation_failed",
                   report_id=report["report_id"], error=str(e))
        obj = None
    finally:
        del raw
        _free_cuda_memory()

    return obj


def run_stage1(manifest: dict) -> dict:
    n_ok, n_fail = 0, 0
    for report in manifest["reports"]:
        profile = generate_reference_profile(report)
        if profile is not None:
            n_ok += 1
        else:
            n_fail += 1
        del profile
        _free_cuda_memory()
    log_event("stage1_profile", "INFO", "stage1_complete", n_ok=n_ok, n_fail=n_fail)
    return {"n_ok": n_ok, "n_fail": n_fail}


STAGE1_RESULT = run_stage1(STAGE0_MANIFEST)
print(f"Stage 1 complete: {STAGE1_RESULT['n_ok']} profiles ok, "
      f"{STAGE1_RESULT['n_fail']} failed.")

[2026-07-05T09:58:05.635360+00:00] [INFO] [stage1_profile] profile_generated :: {'report_id': '5e6d98d4006f9eec322846dd'}
[2026-07-05T09:58:07.933713+00:00] [WARN] [stage1_profile] input_truncated_for_memory :: {'report_id': '35ed2dd1c7889989991747ee', 'max_tokens': 4000}
[2026-07-05T10:03:29.291878+00:00] [INFO] [stage1_profile] profile_generated :: {'report_id': '35ed2dd1c7889989991747ee'}
[2026-07-05T10:03:30.196628+00:00] [WARN] [stage1_profile] input_truncated_for_memory :: {'report_id': '4772a1d980f05b15994e318d', 'max_tokens': 4000}
[2026-07-05T10:05:25.415977+00:00] [INFO] [stage1_profile] profile_generated :: {'report_id': '4772a1d980f05b15994e318d'}
[2026-07-05T10:05:27.825876+00:00] [WARN] [stage1_profile] input_truncated_for_memory :: {'report_id': 'e31f256a58fed7bda10e69ac', 'max_tokens': 4000}
[2026-07-05T10:08:04.005176+00:00] [INFO] [stage1_profile] profile_generated :: {'report_id': 'e31f256a58fed7bda10e69ac'}
[2026-07-05T10:08:04.864303+00:00] [WARN] [stage1_profile] 

In [ ]:
# Stage 1.1b: Version-aware Reference Profile adapter. Any profile already
# on Drive under the old v1 (nested-object) schema remains valid ground
# truth and is NOT regenerated. Instead, every downstream consumer (Stage
# 3/5/6 metric functions) reads profiles through get_profile_field() rather
# than indexing the JSON directly, so both schema versions present a
# uniform "flat list of strings" view without touching Stage 3/5/6 code.
_V1_NESTED_FIELDS = {
    "numerical_values": ["value", "unit", "context"],
    "kpis": ["name", "value", "unit", "period"],
    "financial_metrics": ["name", "value", "unit"],
    "environmental_metrics": ["name", "value", "unit"],
    "targets": ["description", "target_value", "target_year"],
    "dates_events": ["date", "event"],
}


def _flatten_v1_item(item, key_order: list) -> str:
    if isinstance(item, str):
        return item
    if isinstance(item, dict):
        parts = [str(item.get(k, "")) for k in key_order if item.get(k)]
        return ", ".join(parts)
    return str(item)


def get_profile_field(profile: dict, field: str) -> list:
    """Uniform accessor: returns field's contents as a flat list of strings
    regardless of whether `profile` is schema v1 (nested per-item dicts) or
    v2 (flat strings already). Every Stage 3/5/6 function that currently
    does `profile.get(field, [])` directly on a structured field should be
    switched to `get_profile_field(profile, field)` for version safety.
    Scalar fields (company/year/industry/executive_summary) are unaffected
    across versions and can still be read directly.
    """
    raw = profile.get(field, [])
    if not isinstance(raw, list):
        return []
    if field in _V1_NESTED_FIELDS:
        key_order = _V1_NESTED_FIELDS[field]
        return [_flatten_v1_item(item, key_order) for item in raw if item]
    # Already-flat fields (esg_topics, entities, risks, etc.) are identical
    # in shape across v1 and v2 — pass through, coercing to str defensively.
    return [item if isinstance(item, str) else str(item) for item in raw if item]


def get_profile_schema_version(profile: dict) -> str:
    return profile.get("_meta", {}).get("schema_version", "1.0")


log_event("stage1_adapter", "INFO", "version_adapter_ready",
          nested_v1_fields=list(_V1_NESTED_FIELDS.keys()))

In [ ]:
# Stage 1.2: Stage 1 QA pass — a lightweight, transparent (non-LLM) sanity
# check over generated profiles, logged for auditability. This is not a
# scientific metric; it only catches degenerate outputs (empty profiles,
# suspiciously short executive summaries) so they can be flagged before
# being used as ground truth in every later stage.
def qa_check_profile(report_id: str, profile: dict) -> list:
    issues = []
    if not profile.get("company"):
        issues.append("missing_company")
    if len(profile.get("executive_summary", "")) < 30:
        issues.append("executive_summary_too_short")
    array_fields = [
        "esg_topics", "entities", "organizations", "kpis", "financial_metrics",
        "environmental_metrics", "risks", "opportunities", "commitments",
    ]
    empty_arrays = [f for f in array_fields if len(profile.get(f, [])) == 0]
    if len(empty_arrays) >= len(array_fields) // 2:
        issues.append(f"many_empty_arrays:{empty_arrays}")
    return issues


def run_stage1_qa(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        p_path = profile_path(report["report_id"])
        if not os.path.exists(p_path):
            rows.append({"report_id": report["report_id"], "company": report["company"],
                         "status": "missing", "issues": "profile_generation_failed"})
            continue
        with open(p_path, "r") as f:
            profile = json.load(f)
        issues = qa_check_profile(report["report_id"], profile)
        status = "ok" if not issues else "flagged"
        if issues:
            log_event("stage1_qa", "WARN", "profile_flagged",
                       report_id=report["report_id"], issues=issues)
        rows.append({"report_id": report["report_id"], "company": report["company"],
                     "status": status, "issues": ";".join(issues)})
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["reference_profiles"], "_stage1_qa_report.csv")
    df.to_csv(out_csv, index=False)
    log_event("stage1_qa", "INFO", "qa_report_written", path=out_csv,
              n_ok=int((df["status"] == "ok").sum()),
              n_flagged=int((df["status"] == "flagged").sum()),
              n_missing=int((df["status"] == "missing").sum()))
    return df


STAGE1_QA_DF = run_stage1_qa(STAGE1_SAMPLE)
STAGE1_QA_DF["status"].value_counts()

In [ ]:
# Stage 2.0: Shared compression utilities — token-budget enforcement is
# implemented programmatically here (not trusted to the prompt alone), used
# identically by baseline/A/B/C so ratio-tolerance logic lives in one place.
COMPRESSED_DIR = {
    "baseline": CONFIG["PATHS"]["compressed_baseline"],
    "A": CONFIG["PATHS"]["compressed_A"],
    "B": CONFIG["PATHS"]["compressed_B"],
    "C": CONFIG["PATHS"]["compressed_C"],
}


def target_token_range(n_tokens_original: int) -> tuple:
    target = n_tokens_original * CONFIG["COMPRESSION_TARGET_RATIO"]
    tol = n_tokens_original * CONFIG["COMPRESSION_TOLERANCE"]
    return max(1, int(target - tol)), int(target + tol)


def compressed_path(variant: str, report_id: str) -> str:
    return os.path.join(COMPRESSED_DIR[variant], f"{report_id}.json")


def _trim_to_token_budget(text: str, max_tokens: int) -> str:
    """Hard trim as a last resort so the tolerance constraint is NEVER
    violated in the saved artifact, even if generation overshoots. Trims at
    the token level via the canonical tokenizer, then decodes back to text."""
    _, tok = get_model_and_tokenizer()
    ids = tok(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text
    trimmed_ids = ids[:max_tokens]
    return tok.decode(trimmed_ids, skip_special_tokens=True)


def save_compression_result(variant: str, report: dict, compressed_text: str,
                             method: str, n_regen_attempts: int, extra: dict = None) -> dict:
    n_tok = count_tokens(compressed_text)
    lo, hi = target_token_range(report["n_tokens_original"])
    result = {
        "report_id": report["report_id"],
        "company": report["company"],
        "variant": variant,
        "method": method,
        "compressed_text": compressed_text,
        "n_tokens_original": report["n_tokens_original"],
        "n_tokens_compressed": n_tok,
        "actual_ratio": n_tok / max(1, report["n_tokens_original"]),
        "target_ratio": CONFIG["COMPRESSION_TARGET_RATIO"],
        "tolerance": CONFIG["COMPRESSION_TOLERANCE"],
        "within_tolerance": lo <= n_tok <= hi,
        "n_regen_attempts": n_regen_attempts,
    }
    if extra:
        result.update(extra)
    out_path = compressed_path(variant, report["report_id"])
    tmp = out_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(result, f, indent=2, default=str)
    os.replace(tmp, out_path)
    log_event("stage2_compress", "INFO", "compression_saved",
              variant=variant, report_id=report["report_id"],
              n_tok=n_tok, target_range=(lo, hi), within_tolerance=result["within_tolerance"])
    return result


def load_or_none(variant: str, report_id: str):
    p = compressed_path(variant, report_id)
    if os.path.exists(p):
        with open(p, "r") as f:
            return json.load(f)
    return None

In [ ]:
# Stage 2.1: Compression Baseline (control) — naive truncation, no LLM
# involved. Keeps the first target-ratio fraction of tokens verbatim. This
# is the floor that A/B/C must beat on every metric.
def compress_baseline(report: dict) -> dict:
    cached = load_or_none("baseline", report["report_id"])
    if cached is not None:
        return cached

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        text = f.read()

    lo, hi = target_token_range(report["n_tokens_original"])
    target_mid = (lo + hi) // 2
    truncated = _trim_to_token_budget(text, target_mid)

    return save_compression_result(
        "baseline", report, truncated,
        method="naive_first_n_tokens_truncation",
        n_regen_attempts=0,
    )


def run_stage2_baseline(manifest: dict) -> int:
    n = 0
    for report in manifest["reports"]:
        compress_baseline(report)
        n += 1
    log_event("stage2_compress", "INFO", "baseline_compression_complete", n=n)
    return n


_n_baseline = run_stage2_baseline(STAGE0_MANIFEST)
print(f"Baseline compression complete for {_n_baseline} reports.")

In [ ]:
# Stage 2.2: Compression A (simple direct prompt) and Compression B
# (few-shot). Both enforce the token budget programmatically: after each
# generation, actual token count is measured; if outside tolerance, the
# model is re-prompted with explicit feedback ("too long by N tokens" /
# "too short by N tokens") up to COMPRESSION_MAX_REGEN_ATTEMPTS times, then
# hard-trimmed as a last-resort guarantee.
COMPRESSION_A_PROMPT_VERSION = "v1"
COMPRESSION_B_PROMPT_VERSION = "v1"

_COMPRESSION_SYSTEM_PROMPT = (
    "You compress business reports while preserving as much factual "
    "information as possible. You never add information that is not in "
    "the source text."
)

# Two fixed few-shot exemplars for Compression B — short, generic, and
# schema-agnostic on purpose, so they teach the STYLE of dense factual
# compression without leaking Reference-Profile-schema knowledge (which
# would make B functionally identical to C).
_FEW_SHOT_EXAMPLES = [
    {
        "original": (
            "GreenCorp reduced Scope 1 emissions by 18% year-over-year, from "
            "120,000 tCO2e in 2023 to 98,400 tCO2e in 2024, driven by a "
            "transition to renewable electricity at its three largest "
            "manufacturing sites. The company also announced a new supplier "
            "code of conduct requiring all Tier 1 suppliers to disclose "
            "emissions data by 2026."
        ),
        "compressed": (
            "GreenCorp: Scope 1 emissions -18% YoY (120,000 -> 98,400 tCO2e, "
            "2023-2024), driven by renewable electricity at 3 largest plants. "
            "New supplier code of conduct: Tier 1 suppliers must disclose "
            "emissions by 2026."
        ),
    },
    {
        "original": (
            "The board's Audit Committee met four times in fiscal year 2024 "
            "and oversaw the implementation of a new whistleblower hotline, "
            "which received 42 reports, 12 of which were substantiated and "
            "led to corrective action. Employee engagement scores rose from "
            "68% to 74% following the changes."
        ),
        "compressed": (
            "FY2024: Audit Committee met 4x; oversaw new whistleblower "
            "hotline (42 reports, 12 substantiated, corrective action taken). "
            "Employee engagement 68% -> 74% post-changes."
        ),
    },
]


def _build_compression_a_prompt(text: str, target_tokens: int) -> str:
    return (f"Compress the following report to approximately {target_tokens} tokens "
            f"while preserving the most important factual content (numbers, KPIs, "
            f"commitments, risks, entities). Do not add commentary.\n\n"
            f"REPORT:\n\"\"\"\n{text}\n\"\"\"\n\nCompressed version:")


def _build_compression_b_prompt(text: str, target_tokens: int) -> str:
    examples_block = "\n\n".join(
        f"EXAMPLE ORIGINAL:\n{ex['original']}\n\nEXAMPLE COMPRESSED:\n{ex['compressed']}"
        for ex in _FEW_SHOT_EXAMPLES
    )
    return (f"Study these examples of dense, factual report compression:\n\n"
            f"{examples_block}\n\n"
            f"Now compress the following report the same way, to approximately "
            f"{target_tokens} tokens, preserving KPIs, numbers, commitments, risks, "
            f"and named entities. Do not add commentary.\n\n"
            f"REPORT:\n\"\"\"\n{text}\n\"\"\"\n\nCompressed version:")


def _compress_with_regen_loop(report: dict, prompt_builder, prompt_version: str,
                               variant: str, method_name: str) -> dict:
    cached = load_or_none(variant, report["report_id"])
    if cached is not None:
        return cached

    with open(report["extracted_text_path"], "r", encoding="utf-8") as f:
        text = f.read()

    lo, hi = target_token_range(report["n_tokens_original"])
    target_mid = (lo + hi) // 2
    feedback = ""
    result_text = None

    for attempt in range(1, CONFIG["COMPRESSION_MAX_REGEN_ATTEMPTS"] + 1):
        prompt = prompt_builder(text, target_mid) + feedback
        out = generate(
            prompt, system=_COMPRESSION_SYSTEM_PROMPT,
            max_new_tokens=int(hi * 1.5) + 64,
            cache_dir=CONFIG["PATHS"]["gen_cache"],
            cache_extra=f"{variant}:{prompt_version}:attempt{attempt}",
        )
        n_tok = count_tokens(out)
        if lo <= n_tok <= hi:
            result_text = out
            break
        diff = n_tok - target_mid
        direction = "too long" if diff > 0 else "too short"
        feedback = (f"\n\nYour previous output was {direction} by "
                    f"{abs(diff)} tokens (target ~{target_mid} tokens, "
                    f"range {lo}-{hi}). Rewrite it to fit the target length "
                    f"while keeping the same factual content.")
        result_text = out  # keep last attempt in case all attempts miss

    final_n_attempts = attempt
    if not (lo <= count_tokens(result_text) <= hi):
        result_text = _trim_to_token_budget(result_text, target_mid)
        log_event("stage2_compress", "WARN", "hard_trim_applied",
                   variant=variant, report_id=report["report_id"])

    return save_compression_result(
        variant, report, result_text, method=method_name,
        n_regen_attempts=final_n_attempts,
    )


def compress_A(report: dict) -> dict:
    return _compress_with_regen_loop(
        report, _build_compression_a_prompt, COMPRESSION_A_PROMPT_VERSION,
        "A", "direct_prompt",
    )


def compress_B(report: dict) -> dict:
    return _compress_with_regen_loop(
        report, _build_compression_b_prompt, COMPRESSION_B_PROMPT_VERSION,
        "B", "few_shot_prompt",
    )


def run_stage2_ab(manifest: dict) -> dict:
    n_a, n_b = 0, 0
    for report in manifest["reports"]:
        compress_A(report); n_a += 1
        compress_B(report); n_b += 1
    log_event("stage2_compress", "INFO", "A_B_compression_complete", n_a=n_a, n_b=n_b)
    return {"n_a": n_a, "n_b": n_b}


STAGE2_AB_RESULT = run_stage2_ab(STAGE0_MANIFEST)
print(f"Compression A/B complete: {STAGE2_AB_RESULT}")

In [ ]:
# Stage 2.3: Compression C — expert structured-extraction prompting that
# explicitly anchors on the Reference Profile schema fields, instructing the
# model to preserve business/ESG-critical information categories rather than
# compressing generically. This is the strategy expected to best beat the
# baseline on KPI/entity/numerical preservation metrics (Stage 3), since it
# is schema-aware rather than purely length-driven.
COMPRESSION_C_PROMPT_VERSION = "v1"


def _build_compression_c_prompt(text: str, target_tokens: int) -> str:
    priority_fields = (
        "KPIs and financial/environmental metrics (with exact values and units), "
        "named entities and organizations, risks, opportunities, commitments and "
        "targets (with target years), governance and social information, key dates "
        "and events, and applicable regulations/frameworks/standards"
    )
    return (
        f"You are compressing an ESG/sustainability report for an investor/analyst "
        f"audience. Produce a dense compressed version of approximately "
        f"{target_tokens} tokens that PRIORITIZES preserving, in order of "
        f"importance:\n"
        f"1. {priority_fields}\n"
        f"2. Executive-summary-level narrative context, only as space allows.\n\n"
        f"Rules:\n"
        f"- Prefer terse, information-dense phrasing (e.g. 'Scope 1: -18% YoY, "
        f"120,000->98,400 tCO2e') over full sentences.\n"
        f"- Never drop a specific number, unit, date, or named entity to save space "
        f"if a vaguer sentence could be shortened instead.\n"
        f"- Do not add any information not present in the source text.\n\n"
        f"REPORT:\n\"\"\"\n{text}\n\"\"\"\n\nCompressed version:"
    )


def compress_C(report: dict) -> dict:
    return _compress_with_regen_loop(
        report, _build_compression_c_prompt, COMPRESSION_C_PROMPT_VERSION,
        "C", "expert_structured_extraction_prompt",
    )


def run_stage2_c(manifest: dict) -> int:
    n = 0
    for report in manifest["reports"]:
        compress_C(report)
        n += 1
    log_event("stage2_compress", "INFO", "C_compression_complete", n=n)
    return n


STAGE2_C_RESULT = run_stage2_c(STAGE0_MANIFEST)
print(f"Compression C complete for {STAGE2_C_RESULT} reports.")

In [ ]:
# Stage 2.4: Stage 2 wrap-up — consolidated compression summary table across
# all variants (baseline/A/B/C) for quick sanity inspection before moving to
# Stage 3 evaluation. Flags any report/variant combination still outside
# tolerance after the hard-trim safeguard (should be empty by construction,
# but checked explicitly for auditability).
def build_stage2_summary(manifest: dict) -> "pd.DataFrame":
    rows = []
    for report in manifest["reports"]:
        for variant in ["baseline", "A", "B", "C"]:
            res = load_or_none(variant, report["report_id"])
            if res is None:
                rows.append({
                    "report_id": report["report_id"], "company": report["company"],
                    "variant": variant, "status": "missing",
                })
                continue
            rows.append({
                "report_id": report["report_id"],
                "company": report["company"],
                "variant": variant,
                "status": "ok" if res["within_tolerance"] else "OUT_OF_TOLERANCE",
                "n_tokens_original": res["n_tokens_original"],
                "n_tokens_compressed": res["n_tokens_compressed"],
                "actual_ratio": round(res["actual_ratio"], 4),
                "n_regen_attempts": res["n_regen_attempts"],
            })
    df = pd.DataFrame(rows)
    out_csv = os.path.join(CONFIG["PATHS"]["evaluations"], "stage2_compression_summary.csv")
    df.to_csv(out_csv, index=False)

    n_bad = int((df["status"] == "OUT_OF_TOLERANCE").sum())
    n_missing = int((df["status"] == "missing").sum())
    log_event("stage2_compress", "INFO" if (n_bad == 0 and n_missing == 0) else "WARN",
              "stage2_summary_written", path=out_csv,
              n_out_of_tolerance=n_bad, n_missing=n_missing, n_rows=len(df))
    return df


STAGE2_SUMMARY_DF = build_stage2_summary(STAGE0_MANIFEST)
print(f"Stage 2 complete. Summary rows: {len(STAGE2_SUMMARY_DF)}")
STAGE2_SUMMARY_DF.groupby("variant")["actual_ratio"].describe()